In [61]:
# Import necessary libraries for the lab

import numpy as np  # For numerical computations
import matplotlib.pyplot as plt  # For plotting
import random  # For random number generation

from statsmodels.duration.hazard_regression import PHReg  # Though not used in this notebook

# Histograms and the binomial distribution

Histograms provide a visual representation of the empirical distribution of a data set.

In this lab, we will simulate numerical experiments with random number generators to illustrate how histograms can be used to infer the true probability distribution of our random variables and the impact of finite data sets on our estimates.

For simplicity, we will start with an experiment with discrete random variables (the roll of a dice). In the second part of the lab, we will move on to continuous random variables.

## Part 1: Discrete Random Variables : The roll of a dice

Consider a discrete random variable \(Y\) that can take any of the possible values in $\{y_1,\ldots,y_n\}$. To estimate the probability of each value, $p_n$, we can count the frequency with which each $y_i$ occurs in our data set. This intuitive idea is justified by the law of large numbers, but we can easily illustrate it here by modeling the counting process with the binomial distribution.

For clarity, we assume that the true probability of observing $y_i$ is $p_i$.

The process of checking whether our random variable $y$ matches a given value $y_i$ (e.g., a particular digit in a die) can be treated as a Bernoulli trial assigned to the random variable $X_i$, where "success" ($x_i=0$) means that $Y=y_i$, and "failure" ($x_i=1$) means that $Y|\neq y_i$. Since we are describing histograms, we refer to the event $Y=y_i$, as $Y$ being in bin $i$.

In a set of $N$ trials (e.g., $N$ dice rolls), the number of times we observe $Y=y_i$ is also a random variable

$$K_i=\sum_{n=1}^N X_{i,n}.$$

As we discussed in class, $K_i$ follows a binomial distribution:
$$ P(K_i=k) =B(k,p,N)= \binom{N}{k} p_i^k (1-p_i)^{N-k} $$

**Exercises (analytical)**
1. Calculate the expected number of observations in bin \(x_i\) as a function of $p_i$ and $N$.
2. Determine the expected standard deviation (or error) from this theoretical mean for Bin $i$.

**Note:** Sections labelled as "Extension" rely on material beyond Lessons 0–1.

In [62]:
# obtain E[k] and Var[k] as a function of N and p
def compute_mu(N, p):
    return N * p


def compute_sigma2(N, p):
    return N * p * (1 - p)

### Question 1 — Counting repetitions (Lessons 0–1)


Our numerical experiment will mimic a dice rolling experiment. To do this, we will randomly generate dice numbers with a uniform distribution, count how many times we get each of the 6 digits, and try to estimate their respective probabilities from analysis of the data.

Now let's count how many times I observe a particular outcome in a series of $N$ trials.

**Exercise (numerical)**

3. We simulate rolling a die by uniformly generating a random number between 1 and 6, i.e., the probability of getting every possible digit is $p=1/6$. Prepare a function that returns the result of a set of $N$ trials

In [63]:
# We generate N random numbers, each corresponding to the number displayed after a die roll
# the function random.randint(A,B) generates a random number between A and B with uniform probability


def throw_dice(N):  # roll N dices
    return [random.randint(1, 6) for i in range(N)]


# we show the series of numbers obtained with each throw of the dice
print(throw_dice(10))

[2, 1, 6, 5, 1, 2, 6, 1, 4, 3]


4. Now let's count how many times I observe a given outcome in a set of $N$ trials. To do this, follow the steps below:

    a. Define a function to count how many times each number occurs

In [64]:
def count_elements(X, Nmax=6):
    count = np.zeros(Nmax)
    for x in X:
        count[x - 1] += 1

    return count.astype(int)  # return the number of times we find a 1, a 2, ...


N = 10

#Generate a sequences of numbers

X = throw_dice(N)
#we show the sequence
print(X)

# Print the number of times we get each number
print([str(i + 1) + ":" + str(c) for i, c in enumerate(count_elements(X))])

[2, 3, 2, 4, 4, 5, 2, 6, 4, 4]
['1:0', '2:3', '3:1', '4:4', '5:1', '6:1']


b. Plot the results on a bar graph and compare them to the theoretical mean and standard deviation. 


In [65]:
p = 1 / 6.
N = 10

mean = [compute_mu(N, p) for i in range(6)]  # theoretical mean for each number
error = [np.sqrt(compute_sigma2(N, p)) for i in range(6)]  # theoretical standard deviation

# we plot the counts of each number
plt.bar(np.arange(1, 7), count_elements(throw_dice(N)), capsize=5, label='experiment')

#we compare with the theoretical expectation
plt.errorbar(np.arange(1, 7), mean, yerr=error, fmt='o', capsize=5, color='black', label='theoretical')
plt.axhline(y=mean[0], ls='--', color='gray')

plt.ylabel(r'$k$ repetitions')
plt.legend()
plt.show()

/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/453603065.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


c. How does this change if you increase the number of die rolls per experiment?


In [66]:
p = 1 / 6.

for N in [10, 50, 100, 1000, 10000]:
    mean = [compute_mu(N, p) for i in range(6)]  # theoretical mean for each number
    error = [np.sqrt(compute_sigma2(N, p)) for i in range(6)]  # theoretical standard deviation

    # we plot the counts of each number
    plt.bar(np.arange(1, 7), count_elements(throw_dice(N)), capsize=5, label='experiment')

    #we compare with the theoretical expectation
    plt.errorbar(np.arange(1, 7), mean, yerr=error, fmt='o', capsize=5, color='black', label='theoretical')
    plt.axhline(y=mean[0], ls='--', color='gray')

    plt.ylabel(r'$k$ repetitions')
    plt.legend()
    plt.show()


/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/3911765672.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


You can repeat the experiment several times and you will get different results. Most likely, you will get several counts that do not agree the expected value and the expected variation. Does this mean that there is a problem with the random number generator? The general answer is: it depends.

If we know the theoretical distribution, the binomial distribution, we can calculate the probability that it will not match the error

$$(\mu-\sigma > k > \mu+\sigma)=1-P(\mu-\sigma < k < \mu+\sigma)=1-\sum_{k=[\mu-\sigma]}^{[\mu+\sigma]} B(k,N,p)$$

d. For the above values of $N$, estimate the probability that the number of hits within the error is inconsistent with the theoretical values. How many of our counts do we expect to be inconsistent with the expected values? 

In [67]:
from scipy.stats import binom  # we can compute the Binomial distribution

# using the code binom.pmf(k, N, p)


for N in [10, 100, 1000, 10000]:

    sum_prob = 0

    p = 1. / 6
    mean = compute_mu(N, p)
    error = np.sqrt(compute_sigma2(N, p))

    min_k = np.ceil(mean - error).astype(int)
    max_k = np.floor(mean + error).astype(int)
    print(f'min_k={min_k}, max_k={max_k}')

    for k in range(min_k, max_k + 1):
        sum_prob += binom.pmf(k, N, p)

    prob_out = 1 - sum_prob
    print(f'N={N}: prob = {prob_out} out of six: {prob_out * 6}')

min_k=1, max_k=2
N=10: prob = 0.3862787850185859 out of six: 2.3176727101115153
min_k=13, max_k=20
N=100: prob = 0.2815586490593863 out of six: 1.689351894356318
min_k=155, max_k=178
N=1000: prob = 0.30841867638749165 out of six: 1.85051205832495
min_k=1630, max_k=1703
N=10000: prob = 0.3207861195238745 out of six: 1.9247167171432469


### Question 2 — Normalized histogram (Lessons 0–1)


Instead of wondering about how often the die gives a certain number as a result, let us use it to estimate the frequency with which we get each of the numbers, which can be used as an approximation for the real probability $p_n$. The frequency is nothing more than the number of success numbers in my bin divided by the total number of trials. In other words, we now want to describe the statistics of normalized random variable:

$Z_i=\frac{1}{N}\sum_{n=0}^N X_{i,n}=\frac{K_i}{N}$

where $X_i$ is the random variable corresponding to the Bernoilli trial of being in bin $i$.

5. Which is the theoretical mean and the expected standard deviaton of this new normalized variable?



In [68]:
def compute_normalized_mu(N, p):
    return p


def compute_normalized_sigma2(N, p):
    return p * (1 - p) / N

6. Compute the the empirical frequency of appearance of each of the 6 numbers and compare them with the expected value and expected standard deviation. How accurate is this estimation as $N$ grows?

In [69]:
p = 1 / 6.

for N in [10, 100, 1000, 10000, 100000]:
    mean = [compute_normalized_mu(N, p) for i in range(6)]  # mean of Z
    error = [np.sqrt(compute_normalized_sigma2(N, p)) for i in range(6)]  # stardard deviation of Z

    frequency = count_elements(
        throw_dice(N)) / N  # count the number of repetions and normalize it by the number of trials

    plt.bar(np.arange(1, 7), frequency, capsize=5, label='experiment')
    plt.errorbar(np.arange(1, 7), mean, yerr=error, fmt='o', capsize=5, color='black', label='theoretical')

    plt.title(r'$N$=' + str(N))
    plt.axhline(y=1. / 6., ls='--', color='gray')
    plt.ylabel(r'freq$(y_n)$')
    plt.legend()

    plt.show()

/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/3292255661.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Question 3 — Central limit theorem (Lessons 0–1)


The law of large numbers states that $\mu_Z \to p$ as $N\to\infty$, and that $\sigma^2_Z=\frac{\sigma_K^2}{N}$. This ensures that as the number of repetitions $N$ increases, our estimate of the probability of each event becomes finer and finer, but also that the expected fluctuations become smaller and smaller!

The central limit theorem goes even further: it states that $Z$ becomes Gaussian distributed $\mathcal{N}(\mu,\sigma^2)$ as the number of repetitions $N$ increases.

7. We have argued that $K_i=N Z_i$ (and hence $Z$) is distributed as a binomial distribution. We can verify that this is indeed the case by repeating our experiment (rolling $N$ dices) $T=1000$ times. And examine the histogram of the results with the binomial and normal distributions. Follow the steps below to do this:

 a. Now consider repeating the same experiment T times and record the number of repetitions you get for each of the numbers. Since all the numbers have the same probability, we can add up all the results in the same vector of values of k_i.

 b. Calculate a histogram of the ks obtained to estimate the frequency with which we obtain each value.
 Plot k/N against this frequency.

 c. Compare the shape of the histograms as you change $N$
 
 d. For each value of $N$ compare the empirical frequency with the binomial and the normal distribution


In [70]:
def gauss(x, mu, sigma2):
    return 1. / np.sqrt(2 * np.pi * sigma2) * np.exp(-(x - mu) ** 2 / (2 * sigma2))


def count_elements_integer(X):
    Nmin = np.min(X)
    Nmax = np.max(X)

    count = np.zeros(Nmax - Nmin + 1)
    for x in X:
        count[x - Nmin] += 1
    return np.arange(Nmin, Nmax + 1), count.astype(int)


T = 1000

for N in [5, 10, 50, 100, 1000]:

    Results = np.zeros((T, 6))
    for t in range(T):
        Results[t] = count_elements(throw_dice(N))

    pk = Results.reshape(6 * T).astype(
        int)  # Since all numbers have the same probability, we can add up all the statistics of all them for the analysis

    myrange, myhist = count_elements_integer(pk)

    plt.bar(myrange, myhist / (6 * T), label=r'$N$=' + str(N), alpha=0.5)

    # binomial probability
    bi_prob = binom.pmf(myrange, N, 1. / 6.)
    plt.plot(myrange, bi_prob, 'o-', color='green', label='Binomial')

    x = np.linspace(myrange[0], myrange[-1], 1000)
    mu = compute_mu(N, 1. / 6.)
    sigma2 = compute_sigma2(N, 1. / 6.)
    plt.plot(x, gauss(x, mu, sigma2), color='orange', label='Gaussian')

    plt.axvline(x=mu, ls='--', color='black')
    plt.axvline(x=mu - np.sqrt(sigma2), ls='--', lw=3, color='gray')
    plt.axvline(x=mu + np.sqrt(sigma2), ls='--', lw=3, color='gray')
    plt.ylabel(r'freq$(y_n)$')
    plt.legend()

    plt.show()


/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/2728777903.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Question 4 — Area under the Gaussian (Lessons 0–1)


We want to estimate the probability of getting values at certain distance of the expected value.
For this it is convenient to use the **error function**:
 
$$erf(z)=\frac{2}{\sqrt{\pi}}\int_0^z e^{-x^2}$$

Using this function, the cumulative distribution function of the $\mathcal{N}(\mu,\sigma)(x)$ is
$$F(x | \mu, \sigma^2) = \frac{1}{2} \left[ 1 + \text{erf}\left(\frac{x - \mu}{\sigma\sqrt{2}}\right) \right]$$

And the area between two values $a$ and $b$
$$\text{Area between } a \text{ and } b = F(b | \mu, \sigma^2) - F(a | \mu, \sigma^2)$$


(i) Proof these two expressions

(ii) Estimate the probability of  $|k-\mu|>\sigma$, $|k-\mu|>2\sigma$, $|k-\mu|>3\sigma$

In [71]:
from scipy.special import erf


def gaussian_area(a, b):
    return 0.5 * (erf(b / np.sqrt(2)) - erf(a / np.sqrt(2)))


area_under_curve = gaussian_area(-1, 1)
probability_out = 1 - area_under_curve
print(f'Area under the curve: {area_under_curve}')
print(f'Probability of being outside 1 sigma: {probability_out}')

Area under the curve: 0.6826894921370859
Probability of being outside 1 sigma: 0.31731050786291415


## Part 2: Continuous random variables (Lessons 0–1)

Normally, our random variables have continuous support. In such cases, the situation is very similar to the previous one, but now we discretize the support to form bins. Once the bins are defined, everything said before about the binomial distribution applies. As before, we can consider the non-normalized version, where we only count the number of hits in that bin, or the normalized case, where we need to normalize not only by the number of entries, but also by the size of the bin, if we want to obtain a function whose integral is 1.

Now we generate numbers following an exponential distribution

$$
f(x,\lambda) = 
\begin{cases} 
\lambda e^{-\lambda x} & \text{for } x \geq 0 \\
0 & \text{for } x < 0 
\end{cases}
$$

1. Obtain the expected value of $X$ and the Variance.


In [72]:
# Parameters
lamb = 1.0
N = 1000  # Number of random numbers to generate

va = np.random.exponential(1 / lamb, N)


def f(x, lamb):
    return lamb * np.exp(-lamb * x)


def compute_mu(lamb):
    return 1 / lamb


def compute_sigma2(lamb):
    return 1 / lamb ** 2


2. Create a function to obtain a normalized histogram of an array of numbers

3. Which are the expected values for each interval?

In [73]:
# do an histogram
def count_elements(seq, NBins):
    BInf = np.min(seq)
    BSup = np.max(seq)
    δ = (BSup - BInf) / NBins

    bins = np.linspace(BInf + δ / 2, BSup - δ / 2, NBins)
    norm = len(seq) * δ
    hist = np.zeros(NBins)
    for x in seq:
        for i, b in enumerate(bins):
            if b - δ / 2 <= x < b + δ / 2:
                hist[i] += 1
                break

    return hist, norm, bins, δ  # bins gives an array with the mid point in the interval


Nb = 20

h1, norm, bins, δ = count_elements(va, Nb)
print(h1)  # unnormalized
print(h1 / norm)  # normalized

x = bins
# plot the normalized histogram
plt.bar(bins, h1 / norm, width=δ, alpha=0.5, label='empirical')

#theoretical distribution
plt.errorbar(compute_mu(lamb), 1 / lamb, color='black', label='theoretical')
plt.legend()
plt.show()

[485. 229. 136.  65.  41.  21.  10.   3.   7.   1.   1.   0.   0.   0.
   0.   0.   0.   0.   0.   0.]
[0.76337504 0.36043894 0.21405981 0.102308   0.06453274 0.03305335
 0.01573969 0.00472191 0.01101778 0.00157397 0.00157397 0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]


/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/635751881.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


4. Repeat the experiment $T=1000$ times and plot the distribution of the average of $X$, and compare with the Gaussian distribution expected.

In [74]:
T = 10000
N = 1000
Results = np.zeros(T)
for t in range(T):
    Results[t] = np.mean(np.random.exponential(1 / lamb, N))

Nb = 20
h1, norm, bins, δ = count_elements(Results, Nb)
x = bins

# Gaussian dist
mu = compute_mu(lamb)
sigma2 = compute_sigma2(lamb) / N
plt.plot(x, gauss(x, mu, sigma2), color='orange', label='Gaussian')

plt.axvline(x=mu, ls='--', color='black')
plt.bar(bins, h1 / norm, width=δ, alpha=0.5, label='empirical')
plt.legend()
plt.show()

/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/452490161.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part 3: Application to Real-World Data (Extension beyond Lessons 0–1)

In this section, we apply the concepts learned to real datasets. We'll use the heights dataset from the course materials to demonstrate how random variables and distributions work with actual data.

## Loading Real Data

Let's load the heights and weights dataset and explore the distribution of heights.

In [75]:
import pandas as pd

# Load the heights dataset

df = pd.read_csv('../../shared/data/heights_weights_sample.csv')

# Normalize sex labels once and store a canonical value in a new column
df['sex_norm'] = df['sex'].astype(str).str.strip().str.lower().map({
    'm': 'male', 'male': 'male', 'f': 'female', 'female': 'female'
})
# Unknown/other labels -> NA
df.loc[~df['sex_norm'].isin(['male', 'female']), 'sex_norm'] = pd.NA

print(df.head())
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

   id  height_cm  weight_kg sex sex_norm
0   1        172         68   M     male
1   2        158         54   F   female
2   3        181         82   M     male
3   4        165         60   F   female
4   5        177         75   M     male
Dataset shape: (10, 5)
Columns: ['id', 'height_cm', 'weight_kg', 'sex', 'sex_norm']


## Analyzing Heights Distribution (Extension)

Let's examine the distribution of heights for males and females separately. We'll compute summary statistics and visualize the distributions.

In [76]:
# Summary statistics by sex
summary = df.groupby('sex')['height_cm'].agg(['mean', 'std', 'median', 'min', 'max'])
print("Height statistics by sex:")
print(summary)

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (sex, data) in enumerate(df.groupby('sex')):
    axes[i].hist(data['height_cm'], bins=20, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'Height Distribution - {sex.capitalize()}')
    axes[i].set_xlabel('Height (cm)')
    axes[i].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

Height statistics by sex:
      mean       std  median  min  max
sex                                   
F    162.8  4.324350   162.0  158  169
M    177.6  5.458938   177.0  172  185


/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/1717373233.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Additional Exercises

1. **QQ-Plot for Normality**: Create a Q-Q plot to check if the heights data follows a normal distribution. Use scipy.stats.probplot.

2. **Compare Distributions**: Compare the height distributions between males and females. Are they significantly different? Use statistical tests.

3. **Estimate Parameters**: Fit a normal distribution to the heights data and estimate the parameters (mean and standard deviation).

4. **Confidence Intervals**: Compute 95% confidence intervals for the mean height of males and females.

### Solutions for Exercises (Part 3)

Below are step-by-step solutions and runnable code for the exercises. For each exercise we show a short explanation, the code to run, and the resulting outputs/plots for interpretation.

In [77]:
# --- Exercise 4: Confidence intervals (t-based and bootstrap)
import numpy as np
from scipy import stats

# show sex counts for debugging
print('Sex value counts:')
print(df['sex'].value_counts(dropna=False))

# normalize sex labels to common values
sex_norm = df['sex'].astype(str).str.strip().str.lower()
# possible male labels
male_mask = sex_norm.isin(['m', 'male'])
female_mask = sex_norm.isin(['f', 'female'])

male = df.loc[male_mask, 'height_cm'].dropna()
female = df.loc[female_mask, 'height_cm'].dropna()

print(f"Found male: {len(male)} rows, female: {len(female)} rows")

def t_confidence_interval(data, alpha=0.05):
    arr = np.asarray(data)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return (np.nan, np.nan)
    mean = np.mean(arr)
    # For n < 2 (single observation) the sample variance is undefined;
    # we return a degenerate interval (mean, mean) to indicate no estimate of uncertainty.
    if n < 2:
        return (mean, mean)
    se = np.std(arr, ddof=1) / np.sqrt(n)
    df = n - 1
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    return mean - t_crit * se, mean + t_crit * se


def bootstrap_ci(data, n_boot=5000, alpha=0.05, seed=0):
    arr = np.asarray(data)
    arr = arr[~np.isnan(arr)]
    rng = np.random.default_rng(seed)
    n = len(arr)
    if n == 0:
        return (np.nan, np.nan)
    # vectorized bootstrap sampling for speed
    samples = rng.choice(arr, size=(n_boot, n), replace=True)
    boot_means = samples.mean(axis=1)
    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return lower, upper

# Overall
ci_t_all = t_confidence_interval(df['height_cm'])
ci_boot_all = bootstrap_ci(df['height_cm'])
print(f"Overall 95% CI (t): {ci_t_all}")
print(f"Overall 95% CI (bootstrap): {ci_boot_all}")

# By sex
ci_t_m = t_confidence_interval(male)
ci_boot_m = bootstrap_ci(male)
ci_t_f = t_confidence_interval(female)
ci_boot_f = bootstrap_ci(female)
print(f"Male 95% CI (t): {ci_t_m}")
print(f"Male 95% CI (bootstrap): {ci_boot_m}")
print(f"Female 95% CI (t): {ci_t_f}")
print(f"Female 95% CI (bootstrap): {ci_boot_f}")

Sex value counts:
sex
M    5
F    5
Name: count, dtype: int64
Found male: 5 rows, female: 5 rows
Overall 95% CI (t): (np.float64(163.70638733502796), np.float64(176.69361266497202))
Overall 95% CI (bootstrap): (np.float64(165.1), np.float64(175.5))
Male 95% CI (t): (np.float64(170.8218336818743), np.float64(184.3781663181257))
Male 95% CI (bootstrap): (np.float64(173.4), np.float64(181.8))
Female 95% CI (t): (np.float64(157.4306107089408), np.float64(168.16938929105922))
Female 95% CI (bootstrap): (np.float64(159.6), np.float64(166.2))


**Instructor notes (CI exercise):**
- The t-based interval assumes the sampling distribution of the mean is approximately normal; it uses the sample standard deviation and a t critical value.
- The bootstrap percentile CI is non-parametric and often more robust when distributional assumptions are doubtful; both intervals should be compared for agreement.
- For very small samples (n < 2) the t-based CI is degenerate; interpret intervals cautiously and report sample sizes when presenting results.

### Confidence intervals for the mean (t-based and bootstrap) (Extension)

In [78]:
# --- Exercise 3: Fit normal distributions (overall and by sex) and plot
from scipy.stats import norm

# overall
mu_all = df['height_cm'].mean()
sigma_all = df['height_cm'].std(ddof=1)

# by sex: accept M/F or male/female
male = df.loc[df['sex_norm'] == 'male', 'height_cm'].dropna()
female = df.loc[df['sex_norm'] == 'female', 'height_cm'].dropna()
mu_m, sigma_m = male.mean(), male.std(ddof=1)
mu_f, sigma_f = female.mean(), female.std(ddof=1)

print(f"Overall: mu={mu_all:.2f}, sigma={sigma_all:.2f}")
print(f"Male: mu={mu_m:.2f}, sigma={sigma_m:.2f}")
print(f"Female: mu={mu_f:.2f}, sigma={sigma_f:.2f}")

# Plot histogram + fitted PDFs
x = np.linspace(df['height_cm'].min() - 5, df['height_cm'].max() + 5, 300)
plt.figure(figsize=(8, 5))
plt.hist(df['height_cm'].dropna(), bins=25, density=True, alpha=0.4, label='Data')
plt.plot(x, norm.pdf(x, mu_all, sigma_all), label='Fitted Normal (all)', color='black')
if len(male) > 0:
    plt.plot(x, norm.pdf(x, mu_m, sigma_m), label='Fitted Normal (male)', color='blue', ls='--')
if len(female) > 0:
    plt.plot(x, norm.pdf(x, mu_f, sigma_f), label='Fitted Normal (female)', color='red', ls='--')
plt.legend()
plt.xlabel('Height (cm)')
plt.ylabel('Density')
plt.title('Histogram with fitted normal PDFs')
plt.grid(alpha=0.3)
plt.show()

/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/155715973.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Overall: mu=170.20, sigma=9.08
Male: mu=177.60, sigma=5.46
Female: mu=162.80, sigma=4.32


**Instructor notes (Normal fit exercise):**
- We fit normals using the sample mean and sample standard deviation (ddof=1). This is a simple MLE-based approach for illustrative purposes.
- Overlay the fitted PDFs on the histogram to assess visual agreement; systematic deviations (skew, heavy tails, multimodality) suggest the normal model is inadequate.
- When the normal fit is poor consider transformations (log) or alternative families.

### Fit Normal distributions and overlay PDFs (Extension)

In [79]:
# --- Exercise 2: Statistical comparison and distribution views
from scipy import stats
from scipy.stats import gaussian_kde

# Use canonical normalized column produced when loading data
male = df.loc[df['sex_norm'] == 'male', 'height_cm'].dropna()
female = df.loc[df['sex_norm'] == 'female', 'height_cm'].dropna()

# Welch's t-test
t_stat, p_t = stats.ttest_ind(male, female, equal_var=False)

# Mann-Whitney U test
u_stat, p_u = stats.mannwhitneyu(male, female, alternative='two-sided')

print(f"Welch t-test: t={t_stat:.3f}, p={p_t:.4f}")
print(f"Mann-Whitney U: U={u_stat:.3f}, p={p_u:.4f}")

# Empirical CDF comparison
male_sorted = np.sort(male)
female_sorted = np.sort(female)
male_cdf = np.arange(1, len(male_sorted) + 1) / len(male_sorted)
female_cdf = np.arange(1, len(female_sorted) + 1) / len(female_sorted)

plt.figure(figsize=(8, 5))
plt.step(male_sorted, male_cdf, where='post', label='Male ECDF')
plt.step(female_sorted, female_cdf, where='post', label='Female ECDF')
plt.xlabel('Height (cm)')
plt.ylabel('Empirical CDF')
plt.title('Empirical CDF Comparison: Male vs Female Heights')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# KDE plots (brief introduction as smooth density estimates)
x_min = min(df['height_cm'].min(), df['height_cm'].min()) - 5
x_max = max(df['height_cm'].max(), df['height_cm'].max()) + 5
xgrid = np.linspace(x_min, x_max, 200)

kde_m = None
kde_f = None
if len(male) > 2 and np.std(male) > 0:
    kde_m = gaussian_kde(male)
if len(female) > 2 and np.std(female) > 0:
    kde_f = gaussian_kde(female)

plt.figure(figsize=(8, 5))
plt.hist([male, female], bins=20, density=True, alpha=0.3, label=['Male', 'Female'])
if kde_m is not None:
    plt.plot(xgrid, kde_m(xgrid), label='KDE Male')
if kde_f is not None:
    plt.plot(xgrid, kde_f(xgrid), label='KDE Female')
plt.legend()
plt.title('Height Distributions with KDE Overlay')
plt.xlabel('Height (cm)')
plt.ylabel('Density')
plt.grid(alpha=0.3)
plt.show()

Welch t-test: t=4.752, p=0.0017
Mann-Whitney U: U=25.000, p=0.0079


/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/2706831886.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/2706831886.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Instructor notes (Male vs female comparison):**
- Welch's t-test compares group means allowing unequal variances; Mann–Whitney tests for differences in distributions (ranks) without assuming normality.
- Use p-values in context: with small samples effect sizes and visual overlap (KDE/hist) are often more informative than p-values alone.
- KDEs give a smoothed estimate of density; if KDE fails due to singular data, fall back to histograms or report the warning.

### Compare distributions: male vs female heights (Extension)

In [80]:
# --- Exercise 1: QQ-plot for Normality (overall and by sex)
import scipy.stats as ss

# overall QQ-plot
plt.figure(figsize=(6, 5))
ss.probplot(df['height_cm'].dropna(), dist="norm", plot=plt)
plt.title('QQ-plot: Heights (all) vs Normal')
plt.grid(alpha=0.3)
plt.show()

# QQ-plot by sex (explicit ordering and robust to missing groups)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
groups = {
    'male': df.loc[df['sex_norm'] == 'male', 'height_cm'].dropna(),
    'female': df.loc[df['sex_norm'] == 'female', 'height_cm'].dropna()
}
for ax, (label, series) in zip(axes, groups.items()):
    if len(series) > 0:
        ss.probplot(series, dist="norm", plot=ax)
        ax.set_title(f'QQ-plot: {label.capitalize()}')
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No data for {label}', ha='center')
plt.tight_layout()
plt.show()

/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/205498415.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/rv/9hkk54m92hq2s0j4cm1xnph53d62_8/T/ipykernel_24368/205498415.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Instructor notes (QQ-plot comparison — QQ-plots):**
- QQ-plots compare empirical quantiles to theoretical normal quantiles; points on the diagonal indicate agreement with normality.
- Systematic departures (curvature or tails) indicate skewness or heavy/light tails respectively; compare male/female panels to see group-specific deviations.
- Small sample size will produce noisy QQ-plots; always report sample sizes and inspect raw histograms as well.

<!-- Duplicate solutions header removed to keep notebook flow tidy. -->